# Task 1: Preprocess and Explore the Data

This notebook extracts historical financial data for **TSLA**, **BND**, and **SPY** from yfinance covering the period from January 1, 2015 to June 30, 2026. We perform data cleaning, exploratory data analysis, stationarity testing, and calculate risk metrics.

In [ ]:
import sys
sys.path.append('../')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import fetch_data, clean_data, save_data
from src.eda_utils import (
    calculate_daily_returns, calculate_rolling_stats, detect_outliers,
    run_adf_test, calculate_var, calculate_sharpe_ratio
)

## 1. Extract Historical Financial Data
We download the data using `yfinance` from Jan 1, 2015 to June 30, 2026.

In [ ]:
tickers = ["TSLA", "BND", "SPY"]
raw_data = fetch_data(tickers, start_date="2015-01-01", end_date="2026-06-30")
cleaned_data = clean_data(raw_data)
save_data(cleaned_data, output_dir="../data/processed")

## 2. Basic Data Statistics and Distributions

In [ ]:
for ticker in tickers:
    print(f"\n--- Basic Statistics for {ticker} ---")
    print(cleaned_data[ticker].describe())

## 3. Exploratory Data Analysis (EDA)

### A. Visualizing Closing Price over Time

In [ ]:
plt.figure(figsize=(14, 7))
for ticker in tickers:
    plt.plot(cleaned_data[ticker]['Adj Close'], label=ticker)
plt.title("Adjusted Closing Price Over Time (2015 - 2026)", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Price ($)", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

### B. Daily Percentage Change (Returns)
Observing daily return volatility.

In [ ]:
returns_dict = {}
plt.figure(figsize=(14, 7))
for ticker in tickers:
    returns_dict[ticker] = calculate_daily_returns(cleaned_data[ticker])
    plt.plot(returns_dict[ticker], label=f"{ticker} daily return", alpha=0.6)
plt.title("Daily Percentage Change (Returns) over Time", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Daily Return", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

### C. Volatility Analysis (20-day Rolling Mean & Standard Deviation)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for i, ticker in enumerate(tickers):
    roll_mean, roll_std = calculate_rolling_stats(cleaned_data[ticker], window=20)
    axes[i].plot(cleaned_data[ticker]['Adj Close'], label='Adj Close', color='blue')
    axes[i].plot(roll_mean, label='20d Rolling Mean', color='orange')
    axes[i].set_title(f"{ticker} Price & Rolling Mean", fontsize=12)
    axes[i].legend()
    axes[i].grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
for ticker in tickers:
    _, roll_std = calculate_rolling_stats(cleaned_data[ticker], window=20)
    plt.plot(roll_std, label=f"{ticker} 20d Rolling Volatility")
plt.title("20-day Rolling Volatility (Standard Deviation)", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Volatility", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

### D. Outlier Detection
Identify dates where daily returns are greater than 3 standard deviations away from the mean.

In [ ]:
for ticker in tickers:
    outliers = detect_outliers(returns_dict[ticker], threshold=3.0)
    print(f"\n{ticker} has {len(outliers)} outliers using a 3-standard-deviation threshold.")
    print("Top 5 most extreme outlier days:")
    print(outliers.abs().sort_values(ascending=False).head(5))

## 4. Seasonality and Trend Analysis (ADF Stationarity Test)

We run the Augmented Dickey-Fuller (ADF) test on both closing prices and daily returns.

In [ ]:
for ticker in tickers:
    print(f"\n=== ADF Test for {ticker} ===")
    price_adf = run_adf_test(cleaned_data[ticker]['Adj Close'])
    print(f"Adj Close - ADF Stat: {price_adf['adf_stat']:.4f}, p-value: {price_adf['p_value']:.4e}")
    print(f"  Is stationary? {price_adf['is_stationary']}")
    
    ret_adf = run_adf_test(returns_dict[ticker])
    print(f"Daily Returns - ADF Stat: {ret_adf['adf_stat']:.4f}, p-value: {ret_adf['p_value']:.4e}")
    print(f"  Is stationary? {ret_adf['is_stationary']}")

## 5. Risk Metrics (Value at Risk & Sharpe Ratio)

In [ ]:
for ticker in tickers:
    var_95 = calculate_var(returns_dict[ticker], confidence_level=0.95)
    var_99 = calculate_var(returns_dict[ticker], confidence_level=0.99)
    sharpe = calculate_sharpe_ratio(returns_dict[ticker])
    print(f"\n--- Risk Metrics for {ticker} ---")
    print(f"Value at Risk (95% Confidence): {var_95*100:.2f}%")
    print(f"Value at Risk (99% Confidence): {var_99*100:.2f}%")
    print(f"Annualized Sharpe Ratio: {sharpe:.4f}")